# FMC Characteristics

- FMC_SDCLK period $FMC_{SDCLK} = 2*T_{FMC_KER_CK} +- 0.5 ns$
- Data input setup time $t_{su} = 3 ns$
- Data input hold time $t_{h} = 15 ns$
- Address valid time: $t_{d} = 2 ns$

In [36]:
t_cycle = 1 / (166e6)
print(f"t_cycle: {t_cycle}")

t_cycle: 6.024096385542168e-09


## Params FMC Input

In [37]:

trdata_hold_fmc = 1.5e-9
trdata_setup_fmc = 3e-9
twdata_valid_fmc = 2e-9
twdata_hold_fmc = 0.5e-9
twdata_delay_fmc = 0# (time from posedge to dq posedge)

# Address
taddr_valid_fmc = 2e-9

# CS
tcs_valid_fmc = 1.5e-9
tcs_hold_fmc = 0

# WE
twe_valid_fmc = 2e-9
twe_hold_time = 0
# RAS
tras_valid_fmc = 1e-9
tras_hold_fmc  = 0

# CAS
tcas_valid_fmc = 2e-9
tcas_hold_fmc = 0.5e-9

## Params Winbond Input
Speed category: -6

In [38]:
# Data
tdata_setup_sd = 1.5e-9
tdata_hold_sd = 0.8e-9
tread_delay_sd = 5e-9# tAC (time from posedge to dq posedge, choose CL3 -> faster speed

# Address
taddr_setup_sd = 1.5e-9
taddr_hold_sd = 0.8e-9

# CKE
tcke_setup_sd = 1.5e-9
tcke_hold_sd = 0.8e-9

# CMD
tcmd_setup_sd = 1.5e-9
tcmd_hold_sd = 0.8e-9

### Setup time margin

Requirement for the data to be stable before the clock edge.
* Time until the signal gets there, referenced to the previous clock edge
	* EQUATION: T_PROP_TIME = T_CO(MAX, SRC) + T_DATA_TRACE_LENGTH
* Time at which the signal should be there
	* EQUATION: T_REQ = T_PERIOD + T_CLOCK_TRACE_LENGTH - T_SETUP_REQUIRED
* So condition becomes:
	* T_PROP_TIME < T_REQ
	* T_CO(MAX, SRC) + T_DATA_TRACE_LENGTH < T_PERIOD + T_CLOCK_TRACE_LENGTH - T_SETUP_REQUIRED
	* T_PERIODIC - T_CO(MAX, SRC) - (T_DATA_TRACE_LENGTH - T_CLOCK_TRACE_LENGTH) > T_SETUP_REQUIRED

In [41]:
# Choose a data trace length
t_data_trace = 1e-9 #? GET CORRECT DATA TRACE LENGTH
# Choose a clock trace length
t_clk_trace = 1e-9 #? GET CORRECT CLK TRACE LENGTH

### DATA FMC OUT -> DATA SD IN (Write)

In [42]:
# Margin calculation for read [time referenced to previous posedge]
tdata_travel = twdata_delay_fmc + t_data_trace 			# Time to travel from sd phy to fmc phy
tdata_exp = t_cycle + t_clk_trace - tdata_setup_sd 		# Expected time
print(f"travel time: {tdata_travel}, expected time: {tdata_exp}")
margin = tdata_exp - tdata_travel
print(f"Time data margin: {margin * 1e9} ns")

travel time: 1e-09, expected time: 5.524096385542168e-09
Time data margin: 4.524096385542168 ns


### DATA SD OUT -> DATA FMC IN (Read)

In [46]:
# Margin calculation for read [time referenced to previous posedge]
tdata_travel = tread_delay_sd + t_data_trace 			# Time to travel from sd phy to fmc phy
tdata_exp = t_cycle + t_clk_trace - trdata_setup_fmc 	# Expected time
print(f"travel time: {tdata_travel}, expected time: {tdata_exp}")
margin = tdata_exp - tdata_travel
print(f"Time data margin: {margin} ns")

travel time: 6e-09, expected time: 4.024096385542168e-09
Time data margin: -1.975903614457832e-09 ns


### Hold time margin
Requirement for the data to be stable after the clock-edge.

* Time until the data stops being shown, referenced to the previous clock edge.
	* EQUATION: T_PROPAGATION_FINAL_END = (T_PERIODIC + TCO(MIN, SRC)) + T_TRACE_LENGTH
* SO for the condition make sure that:
		* T_PERIODIC + T_HOLD_REQUIRED < T_PROPAGATION_FINAL_END
		* T_PERIODIC + T_HOLD_REQUIRED < T_PERIODIC + TCOO(SRC) + T_TRACE_LENGTH
		* TCO(SRC) + T_TRACE_LENGTH - T_HOLD_REQUIRED > 0

NOTE: everything is reference to the last clock-cycle, 



## SDRAM -> FMC